## CSE3CI Lab 3 : Machine Learning with Scikit Learn
This week, we are going to learn how to use some ML models in scikit learn package on regression tasks. One important step before building a model, is to apply some data preprocessing steps on the given dataset.


we will be working on Vehicle dataset from cardekho. This dataset contains information about used cars listed on www.cardekho.com. Our target is to build a price prediction model with the use of regression models. Thus, our model can help mark a suitable price for a given car.

The datasets consist of several independent variables include:

1. Car_Name: Name of the cars
2. Year: Year of the car when it was bought
3. Selling_Price: Price at which the car is being sold
4. Kms_Driven: Number of Kilometres the car is driven
5. Fuel_Type: Fuel type of car (petrol / diesel / CNG / LPG / electric)
6. Seller_Type: Tells if a Seller is Individual or a Dealer
7. Transmission: Gear transmission of the car (Automatic/Manual)
8. Owner: Number of previous owners of the car.
9. Mileage: mileage of the car
10. Engine: engine capacity of the car
11. Max_power: max power of engine
12. Seats: number of seats in the car

### Load the dataset
use pandas to load the csv file "Car details.csv" provided on LMS, then check dataset length and print the first 5 rows of the dataset

In [51]:
import pandas as pd

dataset = pd.read_csv("Car details.csv")
print("dataset length:", len(dataset))
dataset.head()

dataset length: 7761


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,5.0


### Preprocess the dataset

#### Drop columns that we are not going to use

In [52]:
# drop the columns that we are not going to use, here, we are droping the car name
dataset.drop(['name'], axis=1, inplace=True)
dataset.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,5.0
1,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,5.0
2,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,5.0
3,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,5.0
4,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,5.0


#### Dealing with missing values
##### 1) identify if there is any missing value in the dataset

In [53]:
# check if these is any missing value in the dataset
dataset.isna().sum()

,0
year,0
selling_price,0
km_driven,0
fuel,0
seller_type,0
transmission,0
owner,0
mileage,19
engine,19
max_power,13


##### 2) Drop the rows which has missing values

In [54]:
# dealing with missing values, since we only have very small number of missing values in our dataset, we can just remove it for easy processing
dataset = dataset.dropna()
print("dataset length:", len(dataset))

dataset length: 7742


#### Dealing with duplicated rows
##### 1) Check if there is any duplicated rows in the dataset

In [55]:
# check if these is any duplicated rows
dataset.duplicated().any()

np.True_

##### 2) Remove duplicated rows

In [56]:
# remove duplicate
dataset = dataset.drop_duplicates()
print("dataset length:", len(dataset))

dataset length: 6539


#### Remove units strings from column: mileage, engine and max_power, and then transform the column type to float

In [57]:
# define the function which with inputs of a dataframe and column name, and returns the column which has removed the units string
def remove_unit(df,colum_name) :
    t = []
    for i in df[colum_name]:
        number = str(i).split(' ')[0]
        t.append(number)
    return t

dataset['mileage'] = remove_unit(dataset,'mileage')

# transform the column type to float
dataset['mileage'] = pd.to_numeric(dataset['mileage'])
dataset.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248 CC,74 bhp,5.0
1,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498 CC,103.52 bhp,5.0
2,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497 CC,78 bhp,5.0
3,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396 CC,90 bhp,5.0
4,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298 CC,88.2 bhp,5.0


In [58]:
# do the same for engine and max_power and investigate the dataset
dataset['engine'] = remove_unit(dataset,'engine')
dataset['max_power'] = remove_unit(dataset,'max_power')

dataset['engine'] = pd.to_numeric(dataset['engine'])
dataset['max_power'] = pd.to_numeric(dataset['max_power'])
dataset.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248,74.00,5.0
1,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498,103.52,5.0
2,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497,78.00,5.0
3,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396,90.00,5.0
4,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298,88.20,5.0


#### Adding 'age' feature to know how old the car is and dropping 'year' feature as it is useless now

In [59]:
dataset['age'] = 2025 - dataset['year']

# drop the year column by the function that we used before (hints drop function)
dataset.drop(['year'], axis = 1, inplace = True)

# take a look at the dataset afterwards
dataset.head()

,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats,age
0,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248,74.00,5.0,11
1,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498,103.52,5.0,11
2,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497,78.00,5.0,19
3,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396,90.00,5.0,15
4,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298,88.20,5.0,18


#### Get a summary of numerical columns

In [60]:
dataset.describe()

,selling_price,km_driven,mileage,engine,max_power,seats,age
count,6.539000e+03,6.539000e+03,6539.000000,6539.000000,6539.000000,6539.000000,6539.000000
mean,5.305429e+05,7.269697e+04,19.517721,1430.571953,87.857610,5.433553,11.273589
std,5.132614e+05,5.869317e+04,4.045410,491.703071,31.685995,0.979437,3.810427
min,2.999900e+04,1.000000e+03,0.000000,624.000000,32.800000,2.000000,5.000000
25%,2.500000e+05,3.600000e+04,16.800000,1197.000000,68.000000,5.000000,8.000000
50%,4.250000e+05,6.644400e+04,19.640000,1248.000000,81.860000,5.000000,11.000000
75%,6.500000e+05,1.000000e+05,22.540000,1498.000000,100.000000,5.000000,14.000000
max,1.000000e+07,2.360457e+06,42.000000,3604.000000,400.000000,14.000000,31.000000


#### Handling categorical variables
##### 1) check value count for the categorical variables
including column fuel, seller_type, transmission and owner

In [61]:
# check value count for the categorical variables
print(dataset.fuel.value_counts(),"\n")

#please do the same for seller_type, transmission and owner
print(dataset.seller_type.value_counts(),"\n")
print(dataset.transmission.value_counts(), "\n")
print(dataset.owner.value_counts())

fuel
Diesel    3569
Petrol    2885
CNG         51
LPG         34
Name: count, dtype: int64 

seller_type
Individual          5851
Dealer               661
Trustmark Dealer      27
Name: count, dtype: int64 

transmission
Manual       5973
Automatic     566
Name: count, dtype: int64 

owner
First Owner     4160
Second Owner    1886
Third Owner      493
Name: count, dtype: int64


#### 2) Deal with ordinal variables
transform the strings to numbers

In [62]:
#Ordinal encoding
dataset['owner'] = dataset['owner'].replace({'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3})
dataset.head()

/tmp/ipython-input-4170023319.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['owner'] = dataset['owner'].replace({'First Owner': 1, 'Second Owner': 2, 'Third Owner': 3})


,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats,age
0,450000,145500,Diesel,Individual,Manual,1,23.40,1248,74.00,5.0,11
1,370000,120000,Diesel,Individual,Manual,2,21.14,1498,103.52,5.0,11
2,158000,140000,Petrol,Individual,Manual,3,17.70,1497,78.00,5.0,19
3,225000,127000,Diesel,Individual,Manual,1,23.00,1396,90.00,5.0,15
4,130000,120000,Petrol,Individual,Manual,1,16.10,1298,88.20,5.0,18


#### 3) Deal with nominal variables
transform nominal variable into dummy variables:

In [63]:
dataset = pd.get_dummies(dataset, columns=['fuel', 'seller_type', 'transmission'])
dataset.head()

,selling_price,km_driven,owner,mileage,engine,max_power,seats,age,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,seller_type_Dealer,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Automatic,transmission_Manual
0,450000,145500,1,23.40,1248,74.00,5.0,11,False,True,False,False,False,True,False,False,True
1,370000,120000,2,21.14,1498,103.52,5.0,11,False,True,False,False,False,True,False,False,True
2,158000,140000,3,17.70,1497,78.00,5.0,19,False,False,False,True,False,True,False,False,True
3,225000,127000,1,23.00,1396,90.00,5.0,15,False,True,False,False,False,True,False,False,True
4,130000,120000,1,16.10,1298,88.20,5.0,18,False,False,False,True,False,True,False,False,True


#### Check dataset shape

In [64]:
dataset.shape

(6539, 17)

#### Define the input variables and the target variable
target variable is the selling_price, and input variables are the rest of the columns (you can check the dataset column names and the shape to know which column index you should put here)

In [65]:
array = dataset.values
X = array[:,1:17]
y = array[:,0]

### Split the dataset and normalize data

#### Split the training and testing dataset
Randomly sample the dataset with a random state of 123, use 90% for training and 10% for testinguse

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=123)

#### Apply normalization on both train and testing dataset
One reason this is important is because the variables are multiplied by the model weights. So the scale of the outputs and the scale of the gradients are affected by the scale of the inputs.

Although a model might converge without normalization, normalization makes training much more stable.

In [67]:
from sklearn.preprocessing import MinMaxScaler

# fit scaler on training data
norm = MinMaxScaler().fit(X_train)

# transform training data
X_train_norm = norm.transform(X_train)

# transform testing data
X_test_norm = norm.transform(X_test)

### Train a model

#### Approach 1: Train the model based on entire training dataset and then evaluate the model based on testing dataset

Example of how to build a Linear Regression (LR) model

In [68]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_norm, y_train)

test_score = model.score(X_test_norm, y_test)
print("R2 of LR:", test_score)

R2 of LR: 0.5917399630409526


#### Approach 2: Train the model based on training dataset with cross validation and then evaluate the model based on testing dataset
#####  1) Define a 10 fold cross validation with data shufflling and set the random state with 123
benefits of cross validation: the model can be more generalized, and less prone to be over-fiited. Normally value of k is 5 or 10

In [69]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=10, shuffle=True, random_state=123) #set 10-fold cross validation after shuffle the dataset with random seed 123

##### 2) Run 10-fold cross validation and print the average r-squared score based on the cross validation results
For a regression task, the default evaluation metrics is r squared.

In [70]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

#Basic training of the linear regression model
# define a LR model with default parameter setting
lr = LinearRegression()
# run the previously defined 10-fold validation on the dataset
results = cross_val_score(lr, X_train_norm, y_train, cv=kfold)
# print the averae r squared scores
print("Average R2 of LR:",results.mean())

Average R2 of LR: 0.6301286590968103


### Optimize the LR models with cross validatioin

The parameters that can be applied in grid_params can be found here: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html You can add more values and parameters in the grid_params_lr.

In [71]:
# fine tune parameters for lr model
from sklearn.model_selection import GridSearchCV

grid_params_lr = {
    'fit_intercept': [True, False],
    'n_jobs': [None, -1]
}

lr = LinearRegression()
gs_lr_result = GridSearchCV(lr, grid_params_lr, cv=kfold).fit(X_train_norm, y_train)
print(gs_lr_result.best_score_)

0.6301286590968103


### Evaluate a trained model using testing dataset

In [72]:
# use the best model and evaluate on testing set
test_R2 = gs_lr_result.best_estimator_.score(X_test_norm, y_test)
print("R2 in testing:", test_R2)

R2 in testing: 0.5917399630409526


In [73]:
# check the parameter setting for the best selected model
gs_lr_result.best_params_

{'fit_intercept': True, 'n_jobs': None}

### Predict with a trained model

In [74]:
# predict with the first 5 data points
y_predict = gs_lr_result.best_estimator_.predict(X_test_norm[:5])
print(y_predict)

[1214499.9736509   337037.49493902  551382.04202298  143190.56974847
  498800.41080653]


### Save and load a trained model

In [75]:
import pickle

# Save to file in the current working directory
pkl_filename = "lr_model.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(gs_lr_result.best_estimator_, file)

# Load from file
with open(pkl_filename, 'rb') as file:
    pickle_model = pickle.load(file)

# Calculate the accuracy score and predict target values
score = pickle_model.score(X_test_norm, y_test)
print("R2 score:", score)

R2 score: 0.5917399630409526
